# 1) The AC power flow on a real network: your own Newton–Raphson

## Objectives

At the end of this notebook you should be able to:

- state the **AC power flow equations** for a network of many buses, and count the unknowns by bus type
- implement **Newton–Raphson in polar form** with the complex Jacobian, for any network
- read pandapower's internal data — the **Y-bus**, the specified injections, the bus types and the **bus ordering** — and check your solution against it to 10⁻⁸ p.u.
- recognise **quadratic convergence**, and what its loss tells you
- push the network towards its **loadability limit** and see the power flow stop converging

EE-2.3 solved the power flow for two buses; EE-3.3 built the Y-bus of a real network. This lesson puts them together: a Newton–Raphson solver of your own for a network of any size, checked against pandapower on the **IEEE 14-bus test system** — a standard benchmark derived from a 1960s US network, small enough to read and large enough to need a real solver. It is the first half of the week-15 checkpoint: *own Newton–Raphson matches pandapower to 10⁻⁸ p.u.*

In [1]:
import logging
import warnings

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.networks as pn
import pandapower.topology as top

warnings.filterwarnings("ignore")
logging.getLogger("pandapower").setLevel(logging.ERROR)

### The network

In [2]:
net = pn.case14()
pp.runpp(net, init="flat", numba=False, tolerance_mva=1e-10)          # pandapower's solution, for the check later
print(f"{len(net.bus)} buses, {len(net.line)} lines, {len(net.trafo)} transformers, "
      f"{len(net.gen)} generators + 1 slack, {len(net.load)} loads; "
      f"load {net.load.p_mw.sum():.1f} MW / {net.load.q_mvar.sum():.1f} Mvar")

14 buses, 15 lines, 5 transformers, 4 generators + 1 slack, 11 loads; load 259.0 MW / 73.5 Mvar


In [3]:
graph = top.create_nxgraph(net, respect_switches=False)
kind = pd.Series("PQ (load)", index=net.bus.index)
kind[net.gen.bus] = "PV (generator)"
kind[net.ext_grid.bus] = "slack"
colours = {"slack": "tab:red", "PV (generator)": "tab:green", "PQ (load)": "tab:blue"}
# hand-placed close to the textbook one-line diagram: transmission buses 1-5 below, the lower-voltage area above
xy = {1: (0, 3), 2: (1, 1), 3: (3, 0), 4: (3, 2), 5: (1, 3), 6: (1, 5), 7: (4, 3), 8: (5.2, 3), 9: (4, 4.6),
      10: (3, 5.2), 11: (2, 5.6), 12: (0.4, 6.6), 13: (1.8, 6.9), 14: (3.6, 6.6)}
pos = {b: xy[b + 1] for b in net.bus.index}
fig, ax = plt.subplots(figsize=(7, 4.5))
nx.draw_networkx_edges(graph, pos, ax=ax, edge_color="grey")
for k, c in colours.items():
    nodes = kind.index[kind == k]
    nx.draw_networkx_nodes(graph, pos, nodelist=list(nodes), node_color=c, node_size=260, label=k, ax=ax)
nx.draw_networkx_labels(graph, pos, labels={b: b + 1 for b in net.bus.index}, font_size=8, font_color="white", ax=ax)
ax.legend(fontsize=8, loc="lower right")
ax.set_title("IEEE 14-bus test system (bus numbers as in the literature)")
ax.axis("off")
plt.tight_layout()
plt.show()

### The power flow equations

At every bus i the complex power injected into the network is

S_i = P_i + jQ_i = V_i · conj(I_i) = V_i · conj( Σ_k Y_ik V_k )

with complex voltages V_k = |V_k| e^{jθ_k} and the Y-bus Y. Each bus has four quantities — |V|, θ, P, Q — of which two are specified and two are unknown:

| bus type | specified | unknown | equations used |
|---|---|---|---|
| slack (1) | \|V\|, θ | P, Q | none — computed afterwards |
| PV (generator) | P, \|V\| | θ, Q | the P equation |
| PQ (load) | P, Q | \|V\|, θ | the P and the Q equation |

With 4 PV and 9 PQ buses that is 4 + 2·9 = **22 equations in 22 unknowns** (13 angles, 9 voltage magnitudes). Newton–Raphson solves them by repeated linearisation: compute the mismatch F(x) between the specified injections and those implied by the current voltages, solve J Δx = −F with the Jacobian J, update, repeat.

### What pandapower hands us

pandapower converts its tables into a compact internal model — the Y-bus, the specified injections S_bus in per unit on 100 MVA, and the lists of slack, PV and PQ buses. The internal buses are **reordered**: always map results back through `net._pd2ppc_lookups["bus"]` before comparing.

In [4]:
internal = net._ppc["internal"]
Y = internal["Ybus"].toarray()                      # complex admittance matrix (p.u.)
S_spec = internal["Sbus"]                           # specified injections, generation - load (p.u.)
ref, pv, pq = internal["ref"], internal["pv"], internal["pq"]
order = net._pd2ppc_lookups["bus"]                  # pandapower bus index -> internal position
base_mva = net._ppc["baseMVA"]

V0 = np.ones(len(S_spec), dtype=complex)            # flat start: 1 p.u., 0 degrees ...
for table in (net.gen, net.ext_grid):               # ... except the voltage set points of slack and PV buses
    V0[order[table.bus.to_numpy()]] = table.vm_pu.to_numpy()
print(f"Y-bus {Y.shape}, {np.count_nonzero(Y)} non-zero entries ({np.count_nonzero(Y) / Y.size:.0%}); "
      f"slack {ref}, PV {pv}, PQ {pq} (internal numbering)")

Y-bus (14, 14), 54 non-zero entries (28%); slack [0], PV [1 2 5 7], PQ [ 3  4  6  8  9 10 11 12 13] (internal numbering)


### Newton–Raphson in polar form

The Jacobian follows from differentiating S = diag(V) conj(Y V) with respect to the angles θ and magnitudes |V|. In matrix form, with I = Y V and diag(·) a diagonal matrix:

∂S/∂θ = j diag(V) conj( diag(I) − Y diag(V) ),  ∂S/∂|V| = diag(V) conj( Y diag(V/|V|) ) + conj(diag(I)) diag(V/|V|)

J takes the real parts (P equations) for PV and PQ buses and the imaginary parts (Q equations) for PQ buses, with respect to the angles of PV and PQ buses and the magnitudes of PQ buses — exactly the 22 × 22 system above.

In [5]:
def newton_raphson(Y, S_spec, V0, ref, pv, pq, tol=1e-10, max_iter=20):
    """Solve the AC power flow. Returns the complex bus voltages and the largest mismatch (p.u.) per iteration."""
    V = V0.astype(complex).copy()
    pvpq = np.r_[pv, pq]
    history = []
    for _ in range(max_iter + 1):
        I = Y @ V
        mismatch = V * np.conj(I) - S_spec
        F = np.r_[mismatch[pvpq].real, mismatch[pq].imag]
        history.append(np.abs(F).max())
        if history[-1] < tol:
            return V, history
        Vn = V / np.abs(V)
        dS_dVa = 1j * np.diag(V) @ np.conj(np.diag(I) - Y @ np.diag(V))
        dS_dVm = np.diag(V) @ np.conj(Y @ np.diag(Vn)) + np.conj(np.diag(I)) @ np.diag(Vn)
        J = np.block([[dS_dVa[np.ix_(pvpq, pvpq)].real, dS_dVm[np.ix_(pvpq, pq)].real],
                      [dS_dVa[np.ix_(pq, pvpq)].imag, dS_dVm[np.ix_(pq, pq)].imag]])
        dx = np.linalg.solve(J, -F)
        Va, Vm = np.angle(V), np.abs(V)
        Va[pvpq] += dx[:len(pvpq)]
        Vm[pq] += dx[len(pvpq):]
        V = Vm * np.exp(1j * Va)
    raise RuntimeError(f"no convergence in {max_iter} iterations (mismatch {history[-1]:.1e} p.u.)")


V, history = newton_raphson(Y, S_spec, V0, ref, pv, pq)
print("largest mismatch per iteration (p.u.):", "  ".join(f"{h:.1e}" for h in history))

largest mismatch per iteration (p.u.): 9.2e-01  1.0e-01  7.1e-04  6.0e-08  1.2e-14


In [6]:
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.semilogy(range(len(history)), history, marker="o")
ax.axhline(1e-10, color="grey", ls=":", lw=1)
ax.set(xlabel="iteration", ylabel="largest mismatch (p.u.)", title="Newton–Raphson: quadratic convergence",
       xticks=range(len(history)))
plt.tight_layout()
plt.show()

Four iterations from a flat start. Watch the exponent: 10⁻¹, 10⁻³, 10⁻⁸, 10⁻¹⁴ — once the solver is close, each step roughly **doubles the number of correct digits**. That is quadratic convergence, the signature of Newton's method with a correct Jacobian. The first step is the slowest because the flat start (all angles zero) is far from the solution. The history is also the best test of your own code: a Jacobian with an error in it still converges, sometimes, but linearly — the mismatch shrinks by a roughly constant factor each step — or not at all. If your Newton–Raphson needs fifteen iterations on a healthy network, look for a bug in J.

### The checkpoint: your solver against pandapower

In [7]:
own = pd.DataFrame({"vm_pu": np.abs(V)[order[net.bus.index]],
                    "va_degree": np.degrees(np.angle(V))[order[net.bus.index]]}, index=net.bus.index)
diff_vm = (own.vm_pu - net.res_bus.vm_pu).abs().max()
diff_va = (own.va_degree - net.res_bus.va_degree).abs().max()
assert diff_vm < 1e-8 and np.radians(diff_va) < 1e-8, "the week-15 checkpoint: agreement to 1e-8 p.u."
print(f"largest difference to pandapower: {diff_vm:.1e} p.u. in |V|, {diff_va:.1e} degrees in angle")

S = V * np.conj(Y @ V) * base_mva                                    # solved injections, MVA
slack_mw = S[ref[0]].real
losses = S.real.sum()
print(f"slack generation {slack_mw:.2f} MW (pandapower {net.res_ext_grid.p_mw.iloc[0]:.2f}); "
      f"network losses {losses:.2f} MW = {100 * losses / net.load.p_mw.sum():.1f} % of load "
      f"(pandapower lines + transformers {net.res_line.pl_mw.sum() + net.res_trafo.pl_mw.sum():.2f} MW)")
own.assign(bus=own.index + 1).set_index("bus").T.round(3)

largest difference to pandapower: 2.2e-16 p.u. in |V|, 2.1e-14 degrees in angle
slack generation 232.39 MW (pandapower 232.39); network losses 13.39 MW = 5.2 % of load (pandapower lines + transformers 13.39 MW)


bus          1      2       3       4      5       6       7      8       9   \
vm_pu      1.06  1.045   1.010   1.018  1.020   1.070   1.062   1.09   1.056   
va_degree  0.00 -4.983 -12.725 -10.313 -8.774 -14.221 -13.360 -13.36 -14.939   

bus            10      11      12      13      14  
vm_pu       1.051   1.057   1.055   1.050   1.036  
va_degree -15.097 -14.791 -15.076 -15.156 -16.034  

Your solver and pandapower agree to 10⁻¹⁶ p.u. — machine precision, far inside the checkpoint's 10⁻⁸. That is expected: both solve the same equations with the same Y-bus, so any difference beyond rounding would be a bug in one of them, most often in the bus ordering or the per-unit conversion.

The solution also closes the power balance. The slack bus supplies 232.4 MW, which is whatever the specified generators do not: 259.0 MW of load plus **13.4 MW of losses (5.2 %)** minus the fixed generation elsewhere. Losses are not an input — they come out of the power flow, which is one reason the slack bus exists. The voltages lie between 1.01 and 1.09 p.u., held up by the generators' set points, and the angles fall from 0° at the slack to −16° at bus 14, the bus electrically furthest from it: power flows down the angle differences, as in EE-2.2.

### Towards the limit

In [8]:
def solve_scaled(scale):
    """Own Newton-Raphson with all loads and generator outputs scaled; None if it does not converge."""
    S_scaled = S_spec.copy()
    S_scaled[np.r_[pv, pq]] *= scale                    # every specified injection grows with the load
    try:
        return newton_raphson(Y, S_scaled, V0, ref, pv, pq, max_iter=30)
    except (RuntimeError, np.linalg.LinAlgError):
        return None


rows = {}
for scale in [1.0, 2.0, 3.0, 3.5, 4.0, 4.1, 4.2, 4.5]:
    result = solve_scaled(scale)
    rows[scale] = ({"converged": True, "iterations": len(result[1]) - 1, "lowest_vm_pu": np.abs(result[0]).min()}
                   if result else {"converged": False, "iterations": np.nan, "lowest_vm_pu": np.nan})
stress = pd.DataFrame(rows).T.rename_axis("load scale")
stress

           converged iterations lowest_vm_pu
load scale                                  
1.0             True          4         1.01
2.0             True          4     0.973218
3.0             True          5     0.890251
3.5             True          6     0.832412
4.0             True          7     0.733021
4.1            False        NaN          NaN
4.2            False        NaN          NaN
4.5            False        NaN          NaN

Scaling every load and generator together, the solver needs more iterations as the network is stressed — 4 at base load, 7 at four times the load, when the weakest bus has sagged to 0.73 p.u. — and then fails between 4.0 and 4.1 times the load. This is EE-2.3's loadability limit on a real network: near the nose of the P–V curve the Jacobian becomes nearly singular, Newton's steps grow erratic, and beyond the nose **no solution exists** — voltage collapse, not a numerical accident. In practice you cannot always tell the two apart from a single failed run: a bad starting point or a modelling error can also stop Newton–Raphson. That is why operators study the limit with continuation methods that trace the P–V curve, not with one power flow.

Two caveats make this limit optimistic. The generators here have no reactive-power limits — real machines do, and when one reaches its limit its bus turns from PV into PQ and the voltage drops further (the exercise takes this on). And a factor of four is far beyond anything a real network would be operated at; the useful lesson is the shape of the failure, not the number.

## Check your understanding

**The equations**

1. Why does a PV bus contribute one equation and a PQ bus two? What happens to the count if a generator hits its reactive-power limit?
2. The Y-bus is mostly zeros. What does a zero entry Y_ik mean physically, and why does it matter for solving large networks?

**The solver**

1. How can you tell from the mismatch history that the Jacobian is correct? What would a wrong Jacobian look like?
2. Near the loadability limit the solver needs more iterations and then fails. What is happening to the Jacobian there?